# 02 · Exploratory Data Analysis
Admission rate by acuity (ESI), vital sign distributions, and missingness patterns.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
DATA = "/content/drive/MyDrive/prae/triage_subset.parquet"
df = pd.read_parquet(DATA)
df.shape

In [ ]:
df["admitted"] = (df["disposition"] == "Admit").astype(int)
df["admitted"].mean()

## Admission rate by ESI
Admission rate should fall from ESI 1 (most urgent) to ESI 5.

In [ ]:
df.groupby("esi")["admitted"].agg(["mean", "count"])

## Triage vitals

In [ ]:
df.filter(like="triage_vital_").describe().T.round(1)

## Missingness
About 30% of visits have no triage vitals. Check whether missingness itself is predictive.

In [ ]:
(df.isna().mean()*100).sort_values(ascending=False).head(10).round(1)

In [ ]:
df["vitals_missing"] = df["triage_vital_hr"].isna().astype(int)
df.groupby("vitals_missing")["admitted"].mean()

## Findings

**Target and acuity**
- 560,486 ER visits, about 30% admitted.
- Admission rate falls steeply with ESI (1 = most urgent): ESI 1 is 86%, ESI 2 is 55%, ESI 3 is 29%, ESI 4 is 2.2% and ESI 5 is 0.4%.
- ESI is a very strong signal on its own, but most patients sit in the middle. ESI 2 and 3 together are about 71% of all visits, with admission rates of 55% and 29%. That is where a model has to add something beyond the triage score.
- About 2,500 visits have no ESI recorded.

**Vitals**
- Values look clinically plausible: median heart rate 84, systolic blood pressure 131, respiratory rate 18, oxygen saturation 98%, temperature 98.0°F.
- There are some extreme values (respiratory rate up to 69, oxygen saturation as low as 60, heart rate up to 280), so scaling needs to cope with outliers.
- `triage_vital_o2_device` is a 0/1 flag, and most patients are not on an oxygen device.

**Missing data**
- Vitals are missing for roughly 30% of visits (heart rate, blood pressure, respiratory rate), temperature for 33%, oxygen saturation for 48%, and the oxygen-device flag for 41%.
- The similar missing rates across the vitals suggest they were often not recorded together at triage.
- Arrival mode is missing for 3.9% of visits, and chief-complaint flags for under 1%.
- Because missingness is common and may carry information (a patient triaged in a hurry), later notebooks add a "missing" flag for each vital instead of only filling the gaps.

**Implications for modelling**
- Use triage-time columns only, and a stratified random split to keep the 30% admit rate in every split.
- Fit all imputation and scaling on the training rows only.
- Tune for recall rather than accuracy, since missing a patient who needs a bed costs more than preparing a bed that goes unused.